In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.fft
import numpy as np
import math
from typing import Tuple, List, Dict, Optional
from dataclasses import dataclass, field
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from PIL import Image
import json
import os
from pathlib import Path
from tqdm import tqdm, trange
import re

# =================================================================================
# 1. Configuration - Updated for Enhanced Hierarchical VQA Model
# =================================================================================

@dataclass
class QRARConfig:
    """Configuration for the Hierarchical VQA Q-RAR model"""
    # Input settings
    image_size: int = 224
    in_channels: int = 3
    patch_size: int = 16

    # CNN Stem settings
    stem_channels: List[int] = field(default_factory=lambda: [64, 128])

    # Hierarchical Transformer settings
    depths: List[int] = field(default_factory=lambda: [2, 2, 6, 2])
    num_heads: List[int] = field(default_factory=lambda: [4, 8, 16, 32])
    embedding_dim: int = 128 # Increased for more capacity

    # Core Q-RAR Block settings
    routing_hops: int = 3 # NOVELTY: Set to > 1 for multi-hop reasoning
    ffn_ratio: float = 4.0
    dropout: float = 0.1

    # VQA Settings
    vocab_size: int = 100 # Will be updated by the dataset
    max_question_len: int = 20
    question_embedding_dim: int = 256

    # Classification Head
    num_classes: int = 100 # Will be updated by the dataset
    fusion_hidden_dim: int = 1024 # For Bilinear Fusion Head

    # Training
    aux_loss_weight: float = 0.4 # Weight for hierarchical loss

    def __post_init__(self):
        self.num_stages = len(self.depths)
        assert len(self.num_heads) == self.num_stages, "num_heads must match the number of stages"

    @property
    def num_patches(self) -> int:
        return (self.image_size // self.patch_size) ** 2

# =================================================================================
# 2. Core Complex-Valued and Quantum-Inspired Modules
# =================================================================================

class ComplexLinear(nn.Module):
    """Complex linear layer."""
    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        std = 1.0 / math.sqrt(in_features)
        real_weight = torch.randn(out_features, in_features) * std
        imag_weight = torch.randn(out_features, in_features) * std
        self.weight = nn.Parameter(torch.complex(real_weight, imag_weight))
        self.bias = nn.Parameter(torch.complex(torch.zeros(out_features), torch.zeros(out_features)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.linear(x, self.weight, self.bias)

class ComplexLayerNorm(nn.Module):
    """Layer normalization for complex tensors."""
    def __init__(self, normalized_shape: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.bias = nn.Parameter(torch.zeros(normalized_shape))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        x_norm = (x - mean) / (torch.sqrt(var.abs() + self.eps))
        return self.weight * x_norm + self.bias

class ComplexDropout(nn.Module):
    """Dropout for complex tensors."""
    def __init__(self, p: float = 0.5):
        super().__init__()
        self.p = p

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if not self.training or self.p == 0:
            return x
        mask = torch.rand_like(x.real) > self.p
        scale = 1.0 / (1.0 - self.p)
        return torch.complex(x.real * mask * scale, x.imag * mask * scale)

class AmplitudeAttention(nn.Module):
    """Quantum-inspired amplitude-based attention scoring."""
    def __init__(self, dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)
        self.qkv = ComplexLinear(dim, dim * 3)
        self.proj = ComplexLinear(dim, dim)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        B, N, D = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn_complex = torch.matmul(q, k.transpose(-2, -1).conj()) * self.scale
        attn_scores = torch.abs(attn_complex) ** 2
        attn_weights = F.softmax(attn_scores, dim=-1)

        attn_phase = torch.angle(attn_complex)
        attn_weights_complex = attn_weights * torch.exp(1j * attn_phase)

        out = torch.matmul(attn_weights_complex, v)
        out = out.transpose(1, 2).contiguous().view(B, N, D)
        return self.proj(out), attn_weights

class ComplexFFN(nn.Module):
    """Complex feedforward network with GeGLU activation."""
    def __init__(self, in_features, hidden_features, out_features, dropout):
        super().__init__()
        self.linear1 = ComplexLinear(in_features, hidden_features * 2)
        self.linear2 = ComplexLinear(hidden_features, out_features)
        self.dropout = ComplexDropout(dropout)

    def complex_gelu(self, x: torch.Tensor) -> torch.Tensor:
        return torch.complex(F.gelu(x.real), F.gelu(x.imag))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear1(x)
        x1, x2 = x.chunk(2, dim=-1)
        x = x1 * self.complex_gelu(x2)
        x = self.dropout(x)
        return self.linear2(x)

class QRARBlock(nn.Module):
    """
    NOVELTY: Multi-Hop Quantum Reasoning Block.
    This block iteratively refines features for `routing_hops` iterations,
    simulating a multi-step reasoning process.
    """
    def __init__(self, dim, num_heads, ffn_ratio, dropout, routing_hops):
        super().__init__()
        self.routing_hops = routing_hops
        self.norm1 = ComplexLayerNorm(dim)
        self.attn = AmplitudeAttention(dim, num_heads)
        self.norm2 = ComplexLayerNorm(dim)
        self.ffn = ComplexFFN(dim, int(dim * ffn_ratio), dim, dropout)
        
        # Modules for iterative reasoning/refinement within the block
        self.hop_mixer = ComplexLinear(dim, dim)
        self.hop_norm = ComplexLayerNorm(dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Initial Attention + FFN pass
        attn_out, _ = self.attn(self.norm1(x))
        x = x + attn_out
        ffn_out = self.ffn(self.norm2(x))
        x = x + ffn_out
        
        # Multi-Hop Reasoning Loop (if enabled)
        if self.routing_hops > 1:
            for _ in range(self.routing_hops - 1): # First hop is done above
                # Mix information across tokens for the next hop
                mixed_x = self.hop_mixer(x)
                # Apply a gated residual connection to modulate information flow
                gate = torch.complex(F.gelu(mixed_x.real), F.gelu(mixed_x.imag))
                # The refined representation is a gated version of the original plus the mixed version
                x = self.hop_norm(x + x * gate)

        return x

# =================================================================================
# 3. New Modules for Hierarchical VQA Architecture
# =================================================================================

class TextEncoder(nn.Module):
    """Encodes question text into a vector representation."""
    def __init__(self, config: QRARConfig):
        super().__init__()
        self.embedding = nn.Embedding(config.vocab_size, config.question_embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(config.question_embedding_dim, config.question_embedding_dim, batch_first=True)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, question: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        embedded = self.dropout(self.embedding(question))
        outputs, (hidden, cell) = self.lstm(embedded)
        return outputs, hidden.squeeze(0)

class CNNStem(nn.Module):
    """A lightweight CNN stem for rich feature extraction before patchifying."""
    def __init__(self, in_channels, stem_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, stem_channels[0], kernel_size=7, stride=4, padding=3, bias=False)
        self.norm1 = nn.LayerNorm(stem_channels[0], eps=1e-6, elementwise_affine=True)
        self.relu1 = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(stem_channels[0], stem_channels[1], kernel_size=3, stride=2, padding=1, bias=False)
        self.norm2 = nn.LayerNorm(stem_channels[1], eps=1e-6, elementwise_affine=True)
        self.relu2 = nn.ReLU(inplace=True)
        self.conv3 = nn.Conv2d(stem_channels[1], stem_channels[1], kernel_size=3, stride=2, padding=1, bias=False)
        self.norm3 = nn.LayerNorm(stem_channels[1], eps=1e-6, elementwise_affine=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.relu1(self.norm1(self.conv1(x).permute(0, 2, 3, 1))).permute(0, 3, 1, 2)
        x = self.relu2(self.norm2(self.conv2(x).permute(0, 2, 3, 1))).permute(0, 3, 1, 2)
        x = self.norm3(self.conv3(x).permute(0, 2, 3, 1)).permute(0, 3, 1, 2)
        return x

class ComplexPatchEmbedding(nn.Module):
    """ViT-style patch embedding using CNN stem, outputting complex values."""
    def __init__(self, config: QRARConfig):
        super().__init__()
        self.cnn_stem = CNNStem(config.in_channels, config.stem_channels)
        stem_out_dim = config.stem_channels[-1]
        self.proj_real = nn.Conv2d(stem_out_dim, config.embedding_dim, kernel_size=1)
        self.proj_imag = nn.Conv2d(stem_out_dim, config.embedding_dim, kernel_size=1)
        self.pos_embedding = nn.Parameter(
            torch.complex(
                torch.randn(1, config.num_patches, config.embedding_dim) * 0.02,
                torch.randn(1, config.num_patches, config.embedding_dim) * 0.02
            )
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.cnn_stem(x)
        real_patches = self.proj_real(x).flatten(2).transpose(1, 2)
        imag_patches = self.proj_imag(x).flatten(2).transpose(1, 2)
        complex_patches = torch.complex(real_patches, imag_patches)
        return complex_patches + self.pos_embedding

class PatchMerging(nn.Module):
    """Patch Merging Layer for hierarchical structure."""
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.norm = ComplexLayerNorm(4 * in_dim)
        self.reduction = ComplexLinear(4 * in_dim, out_dim)

    def forward(self, x: torch.Tensor, H: int, W: int) -> Tuple[torch.Tensor, int, int]:
        B, N, C = x.shape
        x = x.view(B, H, W, C)

        pad_h = (2 - H % 2) % 2
        pad_w = (2 - W % 2) % 2
        if pad_h > 0 or pad_w > 0:
            x = F.pad(x.permute(0, 3, 1, 2), (0, pad_w, 0, pad_h)).permute(0, 2, 3, 1)

        H_padded, W_padded = x.shape[1], x.shape[2]
        
        x0 = x[:, 0::2, 0::2, :]
        x1 = x[:, 1::2, 0::2, :]
        x2 = x[:, 0::2, 1::2, :]
        x3 = x[:, 1::2, 1::2, :]
        x = torch.cat([x0, x1, x2, x3], -1).view(B, -1, 4 * C)

        x = self.norm(x)
        x = self.reduction(x)
        return x, H_padded // 2, W_padded // 2

class CrossAttention(nn.Module):
    """Cross-attention mechanism to fuse text and image features."""
    def __init__(self, img_dim, text_dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = img_dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.q_proj = ComplexLinear(img_dim, img_dim)
        self.k_proj = nn.Linear(text_dim, img_dim)
        self.v_proj = nn.Linear(text_dim, img_dim)
        self.proj = ComplexLinear(img_dim, img_dim)

    def forward(self, img_tokens: torch.Tensor, text_tokens: torch.Tensor) -> torch.Tensor:
        B, N_img, D_img = img_tokens.shape
        _, N_text, _ = text_tokens.shape

        q = self.q_proj(img_tokens).view(B, N_img, self.num_heads, self.head_dim).transpose(1, 2)

        k_real = self.k_proj(text_tokens)
        v_real = self.v_proj(text_tokens)
        k = torch.complex(k_real, torch.zeros_like(k_real)).view(B, N_text, self.num_heads, self.head_dim).transpose(1, 2)
        v = torch.complex(v_real, torch.zeros_like(v_real)).view(B, N_text, self.num_heads, self.head_dim).transpose(1, 2)

        attn_complex = (q @ k.transpose(-2, -1).conj()) * self.scale
        attn_scores = torch.abs(attn_complex) ** 2
        attn_weights = F.softmax(attn_scores, dim=-1)

        attn_phase = torch.angle(attn_complex)
        attn_weights_complex = attn_weights * torch.exp(1j * attn_phase)

        out = (attn_weights_complex @ v).transpose(1, 2).reshape(B, N_img, D_img)
        return self.proj(out)

class BilinearFusionHead(nn.Module):
    """
    NOVELTY: Bilinear Fusion Head.
    Fuses image and text features using bilinear pooling for richer, more expressive
    multimodal interaction compared to simple concatenation.
    """
    def __init__(self, img_dim: int, text_dim: int, hidden_dim: int, num_classes: int, dropout: float):
        super().__init__()
        # Image features are complex, so we process real and imag parts (2 * img_dim)
        self.img_proj = nn.Linear(img_dim * 2, hidden_dim)
        self.text_proj = nn.Linear(text_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, img_features: torch.Tensor, text_features: torch.Tensor) -> torch.Tensor:
        # img_features is complex, split into real and imag parts
        img_real_imag = torch.cat([img_features.real, img_features.imag], dim=-1)
        
        img_proj = self.img_proj(img_real_imag)
        text_proj = self.text_proj(text_features)
        
        # Bilinear interaction via element-wise product
        fused = self.dropout(img_proj * text_proj)
        
        logits = self.classifier(fused)
        return logits

# =================================================================================
# 4. The New Hierarchical VQA Q-RAR Model
# =================================================================================

class HierarchicalVQA_QRAR(nn.Module):
    """The complete, novel Hierarchical VQA model using enhanced Q-RAR blocks."""
    def __init__(self, config: QRARConfig):
        super().__init__()
        self.config = config

        # 1. Text Encoder Branch
        self.text_encoder = TextEncoder(config)

        # 2. Image Embedding Branch
        self.patch_embed = ComplexPatchEmbedding(config)

        # 3. Hierarchical Stages of Q-RAR Blocks
        self.stages = nn.ModuleList()
        self.patch_mergers = nn.ModuleList()
        current_dim = config.embedding_dim

        for i in range(config.num_stages):
            stage = nn.ModuleList([
                QRARBlock(
                    dim=current_dim, num_heads=config.num_heads[i],
                    ffn_ratio=config.ffn_ratio, dropout=config.dropout,
                    routing_hops=config.routing_hops
                ) for _ in range(config.depths[i])
            ])
            self.stages.append(stage)

            if i < config.num_stages - 1:
                merger = PatchMerging(current_dim, current_dim * 2)
                self.patch_mergers.append(merger)
                current_dim *= 2

        # 4. Fusion (Cross-Attention) Layer
        self.fusion_stage_idx = 0 # Fuse after stage 0
        fusion_img_dim = config.embedding_dim * 2
        self.cross_attention = CrossAttention(
            img_dim=fusion_img_dim,
            text_dim=config.question_embedding_dim,
            num_heads=config.num_heads[self.fusion_stage_idx + 1]
        )

        # 5. NOVELTY: Hierarchical Auxiliary Loss Head
        # This head provides a supervisory signal at an intermediate layer.
        self.auxiliary_head = nn.Sequential(
            ComplexLayerNorm(fusion_img_dim),
            nn.Linear(fusion_img_dim * 2, config.num_classes)
        )

        # 6. Final Classification Head
        self.final_norm = ComplexLayerNorm(current_dim)
        self.head = BilinearFusionHead(
            img_dim=current_dim, text_dim=config.question_embedding_dim,
            hidden_dim=config.fusion_hidden_dim, num_classes=config.num_classes,
            dropout=config.dropout
        )

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            torch.nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
            if m.weight is not None:
                nn.init.constant_(m.weight, 1.0)

    def forward(self, image: torch.Tensor, question: torch.Tensor) -> Dict[str, torch.Tensor]:
        # 1. Process Text
        text_tokens, question_vec = self.text_encoder(question)

        # 2. Process Image
        img_tokens = self.patch_embed(image)
        H, W = int(img_tokens.shape[1] ** 0.5), int(img_tokens.shape[1] ** 0.5)

        aux_logits = None

        # 3. Pass through Hierarchical Stages with Fusion
        for i in range(self.config.num_stages):
            # Apply the Q-RAR blocks for the current stage
            for block in self.stages[i]:
                img_tokens = block(img_tokens)

            # Perform fusion after the designated fusion stage
            if i == self.fusion_stage_idx:
                # Apply patch merger before fusion to get the right dimensions
                if i < len(self.patch_mergers):
                    img_tokens, H, W = self.patch_mergers[i](img_tokens, H, W)

                # a) Apply Cross-Attention Fusion
                fusion_out = self.cross_attention(img_tokens, text_tokens)
                
                # b) NOVELTY: Quantum-Inspired Interference
                # A learned gate modulates the residual connection, simulating
                # constructive/destructive interference patterns.
                gate = torch.complex(torch.tanh(fusion_out.real), torch.tanh(fusion_out.imag))
                img_tokens = img_tokens + fusion_out * gate

                # c) Calculate Auxiliary Loss
                aux_pooled = self.auxiliary_head[0](img_tokens).mean(dim=1)
                aux_pooled_real_imag = torch.cat([aux_pooled.real, aux_pooled.imag], dim=-1)
                aux_logits = self.auxiliary_head[1](aux_pooled_real_imag)

            # For other stages, just apply patch merging if it's not the last one
            elif i < self.config.num_stages - 1:
                img_tokens, H, W = self.patch_mergers[i](img_tokens, H, W)

        # 4. Final Pooling and Classification
        x_pooled = self.final_norm(img_tokens).mean(dim=1)
        logits = self.head(x_pooled, question_vec)

        return {'logits': logits, 'aux_logits': aux_logits}


# =================================================================================
# 5. Updated Data Handling and Training Pipeline
# =================================================================================

class RealCLEVRDataset(Dataset):
    """Real CLEVR dataset - now with text tokenization."""
    def __init__(self, clevr_root: str, split: str = 'train', transform=None,
                 max_samples: Optional[int] = None, max_q_len: int = 20,
                 question_vocab: Optional[Dict[str, int]] = None):
        self.clevr_root = Path(clevr_root)
        self.split = split
        self.transform = transform
        self.max_q_len = max_q_len

        self.images_dir = self.clevr_root / 'images' / split
        self.questions_file = self.clevr_root / 'questions' / f'CLEVR_{split}_questions.json'

        with open(self.questions_file, 'r') as f:
            data = json.load(f)
        self.questions = data['questions']

        if max_samples:
            self.questions = self.questions[:max_samples]

        if question_vocab is None:
            self.question_vocab, self.answer_vocab = self._build_vocabs()
        else:
            self.question_vocab, self.answer_vocab = question_vocab

        self.vocab_size = len(self.question_vocab)
        self.num_answers = len(self.answer_vocab)

    def _tokenize_question(self, question: str) -> List[int]:
        question = question.lower().replace('?', '').replace(',', '')
        words = re.findall(r'\w+', question)
        tokens = [self.question_vocab.get(w, self.question_vocab['<unk>']) for w in words]
        return tokens

    def _build_vocabs(self) -> Tuple[Dict[str, int], Dict[str, int]]:
        q_words = set(['<pad>', '<unk>'])
        answers = set()
        for q_data in tqdm(self.questions, desc=f"Building {self.split} vocabs"):
            answers.add(str(q_data['answer']).lower())
            q_text = q_data['question'].lower().replace('?', '').replace(',', '')
            for word in re.findall(r'\w+', q_text):
                q_words.add(word)

        q_vocab = {word: i for i, word in enumerate(sorted(list(q_words)))}
        a_vocab = {ans: i for i, ans in enumerate(sorted(list(answers)))}
        return q_vocab, a_vocab

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):
        q_data = self.questions[idx]
        image_path = self.images_dir / q_data['image_filename']
        image = Image.open(image_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        q_tokens = self._tokenize_question(q_data['question'])
        q_padded = np.zeros(self.max_q_len, dtype=np.int64)
        q_len = min(len(q_tokens), self.max_q_len)
        q_padded[:q_len] = q_tokens[:q_len]

        answer_idx = self.answer_vocab.get(str(q_data['answer']).lower(), 0)
        return {
            'image': image, 'question': torch.from_numpy(q_padded),
            'answer': torch.tensor(answer_idx, dtype=torch.long),
        }

def create_clevr_dataloaders(clevr_root: str, batch_size: int = 32,
                           max_train_samples: Optional[int] = None,
                           max_val_samples: Optional[int] = None,
                           num_workers: int = 2, max_q_len: int = 20):
    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.3),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_dataset = RealCLEVRDataset(clevr_root, 'train', train_transform, max_train_samples, max_q_len)
    val_dataset = RealCLEVRDataset(clevr_root, 'val', val_transform, max_val_samples, max_q_len,
                                   question_vocab=(train_dataset.question_vocab, train_dataset.answer_vocab))

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

    return train_loader, val_loader, train_dataset.vocab_size, train_dataset.num_answers

def train_model(clevr_root: str, epochs: int = 20, batch_size: int = 32,
                learning_rate: float = 1e-4, max_train_samples: Optional[int] = 10000,
                max_val_samples: Optional[int] = 2000, save_dir: str = 'h_qrar_checkpoints'):

    save_dir = Path(save_dir)
    save_dir.mkdir(exist_ok=True)

    print("Creating CLEVR dataloaders...")
    train_loader, val_loader, vocab_size, num_classes = create_clevr_dataloaders(
        clevr_root, batch_size, max_train_samples, max_val_samples
    )
    print(f"Vocab size: {vocab_size}, Num answers: {num_classes}")

    config = QRARConfig(
        depths=[2, 2, 6, 2],
        num_heads=[4, 8, 16, 24], # Adjusted for new embedding_dim
        embedding_dim=128,
        dropout=0.1,
        routing_hops=3, # Enable multi-hop reasoning
        vocab_size=vocab_size,
        num_classes=num_classes,
        max_question_len=20
    )

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = HierarchicalVQA_QRAR(config).to(device)
    print(f"Using device: {device}")
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total Trainable Parameters: {total_params:,}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-2)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_acc = 0.0
    for epoch in range(epochs):
        model.train()
        train_loss_total = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [TRAIN]")
        for batch in pbar:
            images = batch['image'].to(device)
            questions = batch['question'].to(device)
            labels = batch['answer'].to(device)

            optimizer.zero_grad()
            outputs = model(images, questions)
            
            # Calculate combined loss using main and auxiliary outputs
            main_loss = criterion(outputs['logits'], labels)
            if outputs['aux_logits'] is not None:
                aux_loss = criterion(outputs['aux_logits'], labels)
                loss = main_loss + config.aux_loss_weight * aux_loss
            else:
                loss = main_loss

            loss.backward()
            optimizer.step()

            train_loss_total += loss.item()
            pbar.set_postfix({'Loss': f'{loss.item():.4f}'})

        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        pbar_val = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [VALIDATE]")
        with torch.no_grad():
            for batch in pbar_val:
                images = batch['image'].to(device)
                questions = batch['question'].to(device)
                labels = batch['answer'].to(device)

                outputs = model(images, questions)
                loss = criterion(outputs['logits'], labels)
                val_loss += loss.item()

                _, predicted = torch.max(outputs['logits'], 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                pbar_val.set_postfix({'Acc': f'{100*correct/total:.2f}%'})

        val_accuracy = 100 * correct / total
        print(f"Epoch {epoch+1} Summary: Train Loss: {train_loss_total/len(train_loader):.4f}, Val Loss: {val_loss/len(val_loader):.4f}, Val Acc: {val_accuracy:.2f}%")

        if val_accuracy > best_val_acc:
            best_val_acc = val_accuracy
            torch.save(model.state_dict(), save_dir / 'best_model_novel.pth')
            print(f"★ New best model saved! Accuracy: {val_accuracy:.2f}%")

        scheduler.step()

# =================================================================================
# 6. Main Execution
# =================================================================================

def main():
    CLEVR_ROOT = "/kaggle/input/clevr-dataset/CLEVR_v1.0"
    if not Path(CLEVR_ROOT).exists():
        print("="*60)
        print("!! CLEVR dataset not found at the specified path !!")
        print("Please update the CLEVR_ROOT variable to the correct directory.")
        print("Skipping training.")
        print("="*60)
        return

    train_model(
        clevr_root=CLEVR_ROOT,
        epochs=50,
        batch_size=32, # Adjust based on GPU memory
        learning_rate=1e-4,
        max_train_samples=70000,
        max_val_samples=15000
    )

if __name__ == "__main__":
    main()

Creating CLEVR dataloaders...


Building train vocabs: 100%|██████████| 70000/70000 [00:00<00:00, 151332.05it/s]


Vocab size: 82, Num answers: 28
Using device: cuda
Total Trainable Parameters: 71,561,592


Epoch 1/50 [TRAIN]:   0%|          | 0/2188 [00:02<?, ?it/s]


RuntimeError: shape '[32, 4, 3, 24, 42]' is invalid for input of size 393216